In [ ]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: Tesla T4
overlay         113G   43G   71G  38% /


In [1]:
import numpy as np, pyarrow.parquet as pq, time
ROOT = '/content/wnn_connectome_starterpack/datasets'
L, NF = 20000, 112
N_TRAIN, N_VALID = 600, 100

def build(src, dst, n, seed, with_mask):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]
    assert len(feat) == NF
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (n, L, NF))
    Y = np.zeros((n, L, 2), np.float32)
    M = np.zeros((n, L), bool) if with_mask else None
    t = time.time()
    for k, i in enumerate(idx):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'] + (['is_scored'] if with_mask else []))
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat])
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if with_mask: M[k] = tb['is_scored'].to_numpy()
        if k % 100 == 0: print(dst, k, f'{time.time() - t:.0f} с', flush=True)
    X.flush(); np.save(f'/content/data/{dst}_Y.npy', Y)
    if with_mask: np.save(f'/content/data/{dst}_M.npy', M)

build('train', 'train', N_TRAIN, 0, False)
build('valid', 'valid', N_VALID, 1, True)

FileNotFoundError: [Errno 2] Failed to open local file '/content/wnn_connectome_starterpack/datasets/train.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
import numpy as np, torch, torch.nn as nn, time
dev = 'cuda'
L, NF, WARM = 20000, 112, 99
Xtr = np.load('/content/data/train_X.npy', mmap_mode='r'); Ytr = np.load('/content/data/train_Y.npy')
Xva = np.load('/content/data/valid_X.npy', mmap_mode='r'); Yva = np.load('/content/data/valid_Y.npy')
Mva = np.load('/content/data/valid_M.npy')
print('train', Xtr.shape, '| valid', Xva.shape, '| доля is_scored', Mva[:, WARM:].mean().round(3))

class Net(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.gru = nn.GRU(NF, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

def loss_fn(p, y, mask):
    y = y.clamp(-2, 2)
    return (y.abs() * (p - y) ** 2)[mask].mean()

@torch.no_grad()
def evaluate(model, X, Y, M, bs=50, ch=2000):
    model.eval()
    S = {k: torch.zeros(2, 6, dtype=torch.float64, device=dev) for k in ('все строки', 'is_scored')}
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)
        y = torch.from_numpy(Y[s:s + bs]).to(dev).clamp(-2, 2)
        m_sc = torch.from_numpy(M[s:s + bs]).to(dev)
        h, outs = None, []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); outs.append(o)
        p = torch.cat(outs, 1).clamp(-2, 2)
        post = torch.zeros(L, dtype=torch.bool, device=dev); post[WARM:] = True
        for name, mask in (('все строки', post[None].expand_as(m_sc)), ('is_scored', m_sc & post[None])):
            for j in (0, 1):
                yy, pp = y[..., j][mask].double(), p[..., j][mask].double(); a = yy.abs()
                S[name][j] += torch.stack([a.sum(), (a*yy).sum(), (a*pp).sum(), (a*yy*yy).sum(), (a*pp*pp).sum(), (a*yy*pp).sum()])
    out = {}
    for k, T in S.items():
        sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
        cov = T[:, 5] / sw - my * mp
        den = (T[:, 3] / sw - my**2) * (T[:, 4] / sw - mp**2)
        c = torch.where(den > 1e-16, cov / den.clamp_min(1e-16).sqrt(), torch.zeros_like(cov)).clamp(-1, 1)
        out[k] = (c.mean().item(), c.cpu().numpy().round(3))
    model.train(); return out

def train(model, epochs, bs=32, ch=1000, lr=2e-3, tag='gru'):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    steps_total = epochs * (len(Xtr) // bs) * (L // ch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps_total, pct_start=0.05)
    rng = np.random.default_rng(0); best = -1
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(len(Xtr)); run = []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev)
            h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1)
                loss = loss_fn(o, y[:, t:t + ch], mask)
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
                run.append(loss.item())
        r = evaluate(model, Xva, Yva, Mva)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | '
              f'valid все строки {r["все строки"][0]:.3f} {r["все строки"][1]} | is_scored {r["is_scored"][0]:.3f} {r["is_scored"][1]}', flush=True)
        if r['is_scored'][0] > best:
            best = r['is_scored'][0]; torch.save(model.state_dict(), f'/content/{tag}_best.pt')
    return model

torch.manual_seed(0)
model = Net().to(dev)
print('параметров:', sum(p.numel() for p in model.parameters()))
model = train(model, epochs=1)           # сначала 1 эпоха: смотрим время и что скор не нулевой

train (600, 20000, 112) | valid (100, 20000, 112) | доля is_scored 0.086
параметров: 192258


/tmp/ipykernel_322/4112716574.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/1 | 27 с | loss 1.1271 | valid все строки 0.361 [0.363 0.36 ] | is_scored 0.529 [0.515 0.543]


In [ ]:
!pip install -q onnxruntime
import sys, time
from types import SimpleNamespace
sys.path.insert(0, 'wnn_connectome_starterpack/baseline')
from solution import PredictionModel

mdl = PredictionModel()
Pb = np.zeros((len(Xva), L, 2), np.float32); t = time.time()
for s in range(len(Xva)):
    Xs = np.ascontiguousarray(Xva[s])
    for st in range(L):
        r = mdl.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= WARM, state=Xs[st]))
        if r is not None: Pb[s, st] = r
    if s % 20 == 0: print(s, f'{time.time() - t:.0f} с', flush=True)

def np_score(P, Y, M):
    post = np.zeros(L, bool); post[WARM:] = True
    out = {}
    for name, mask in (('все строки', np.broadcast_to(post, M.shape)), ('is_scored', M & post)):
        cs = []
        for j in (0, 1):
            y = np.clip(Y[..., j][mask], -2, 2).astype(np.float64); p = np.clip(P[..., j][mask], -2, 2).astype(np.float64)
            w = np.abs(y); sw = w.sum(); my, mp = (w * y).sum() / sw, (w * p).sum() / sw
            cov = (w * (y - my) * (p - mp)).sum(); vy, vp = (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
            cs.append(cov / np.sqrt(vy * vp) if vy > 1e-8 and vp > 1e-8 else 0.0)
        out[name] = (round(float(np.mean(cs)), 3), np.round(cs, 3))
    return out
print('baseline.onnx на наших 100 валидационных последовательностях:', np_score(Pb, Yva, Mva))

0 4 с
20 43 с
40 82 с
60 121 с
80 159 с
baseline.onnx на наших 100 валидационных последовательностях: {'все строки': (0.404, array([0.408, 0.4  ])), 'is_scored': (0.572, array([0.555, 0.589]))}


In [ ]:
import os
from google.colab import drive

# 1. Смонтировать Drive, если ещё не смонтирован
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. Создать папку (если её нет)
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True)

# 3. Сохранить
tag = 'baseline'
torch.save(model.state_dict(), f'/content/drive/MyDrive/wnn/{tag}_best.pt')
print('saved:', f'/content/drive/MyDrive/wnn/{tag}_best.pt')
tag='baseline'
torch.save(model.state_dict(), f'/content/drive/MyDrive/wnn/{tag}_best.pt')

Mounted at /content/drive
saved: /content/drive/MyDrive/wnn/baseline_best.pt


In [ ]:
torch.manual_seed(0); model = Net().to(dev)
model = train(model, epochs=20, tag='gru_w')

эпоха 1/20 | 10 с | loss 1.1665 | valid все строки 0.371 [0.379 0.362] | is_scored 0.540 [0.537 0.543]
эпоха 2/20 | 10 с | loss 1.0751 | valid все строки 0.406 [0.421 0.391] | is_scored 0.559 [0.554 0.564]
эпоха 3/20 | 9 с | loss 1.0408 | valid все строки 0.400 [0.417 0.382] | is_scored 0.544 [0.54  0.548]
эпоха 4/20 | 9 с | loss 1.0342 | valid все строки 0.404 [0.411 0.396] | is_scored 0.550 [0.551 0.549]
эпоха 5/20 | 9 с | loss 1.0157 | valid все строки 0.410 [0.414 0.406] | is_scored 0.573 [0.574 0.572]
эпоха 6/20 | 9 с | loss 1.0051 | valid все строки 0.423 [0.439 0.406] | is_scored 0.589 [0.587 0.59 ]
эпоха 7/20 | 9 с | loss 0.9862 | valid все строки 0.412 [0.411 0.412] | is_scored 0.579 [0.574 0.583]
эпоха 8/20 | 9 с | loss 0.9740 | valid все строки 0.415 [0.42  0.409] | is_scored 0.582 [0.58  0.584]
эпоха 9/20 | 9 с | loss 0.9502 | valid все строки 0.392 [0.408 0.375] | is_scored 0.564 [0.562 0.566]
эпоха 10/20 | 9 с | loss 0.9267 | valid все строки 0.383 [0.417 0.348] | is_scor

In [ ]:
!df -h /content | tail -1
build('train', 'train3k', 3000, 0, False)       # около 5 минут
Xtr = np.load('/content/data/train3k_X.npy', mmap_mode='r')
Ytr = np.load('/content/data/train3k_Y.npy')
torch.manual_seed(0); model = Net().to(dev)
model = train(model, epochs=8, tag='gru_w3k')

overlay         113G   39G   74G  35% /
train3k 0 0 с
train3k 100 11 с
train3k 200 22 с
train3k 300 35 с
train3k 400 47 с
train3k 500 62 с
train3k 600 73 с
train3k 700 84 с
train3k 800 92 с
train3k 900 97 с
train3k 1000 108 с
train3k 1100 122 с
train3k 1200 136 с
train3k 1300 147 с
train3k 1400 162 с
train3k 1500 173 с
train3k 1600 185 с
train3k 1700 198 с
train3k 1800 208 с
train3k 1900 217 с
train3k 2000 227 с
train3k 2100 242 с
train3k 2200 253 с
train3k 2300 264 с
train3k 2400 276 с
train3k 2500 291 с
train3k 2600 303 с
train3k 2700 313 с
train3k 2800 328 с
train3k 2900 344 с
эпоха 1/8 | 143 с | loss 1.0530 | valid все строки 0.401 [0.404 0.399] | is_scored 0.575 [0.568 0.583]
эпоха 2/8 | 140 с | loss 1.0012 | valid все строки 0.439 [0.443 0.434] | is_scored 0.592 [0.588 0.596]
эпоха 3/8 | 138 с | loss 0.9895 | valid все строки 0.433 [0.444 0.421] | is_scored 0.589 [0.59  0.587]
эпоха 4/8 | 135 с | loss 0.9794 | valid все строки 0.444 [0.445 0.443] | is_scored 0.596 [0.589 0.602]
э

In [ ]:
!pip install -q onnx
import torch, torch.nn as nn, numpy as np, os, shutil, sys, time
from types import SimpleNamespace

CKPT = '/content/drive/MyDrive/wnn/gru_w3k_best.pt'     # поправь путь, если в train он другой
if not os.path.exists(CKPT): CKPT = '/content/gru_w3k_best.pt'
net = Net(); net.load_state_dict(torch.load(CKPT, map_location='cpu')); net.eval()

class Step(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, features, hidden_0, hidden_1):
        o, h = self.net(features, torch.cat([hidden_0, hidden_1], 0))
        return o, h[0:1], h[1:2]

os.makedirs('/content/sub', exist_ok=True)
torch.onnx.export(Step(net), (torch.zeros(1, 1, 112), torch.zeros(1, 1, 128), torch.zeros(1, 1, 128)),
                  '/content/sub/model.onnx', input_names=['features', 'hidden_0', 'hidden_1'],
                  output_names=['prediction', 'hidden_0_out', 'hidden_1_out'], opset_version=17, dynamo=False)

# solution.py берём у организаторов, меняем только имя файла модели
src = open('/content/wnn_connectome_starterpack/baseline/solution.py').read().replace('baseline.onnx', 'model.onnx')
open('/content/sub/solution.py', 'w').write(src)
!cd /content/sub && rm -f submission.zip && zip -q submission.zip solution.py model.onnx && ls -lh

# --- проверка эквивалентности и скорости: тот самый solution.py на 100 валидационных последовательностях ---
for m in [k for k in sys.modules if k == 'solution']: del sys.modules[m]
sys.path.insert(0, '/content/sub')
import solution
mdl = solution.PredictionModel()
Pn = np.zeros((len(Xva), L, 2), np.float32); t = time.time(); n_rows = 0
for s in range(len(Xva)):
    Xs = np.ascontiguousarray(Xva[s])
    for st in range(L):
        r = mdl.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= WARM, state=Xs[st]))
        if r is not None: Pn[s, st] = r
    n_rows += L
    if s % 25 == 0: print(s, f'{time.time() - t:.0f} с', flush=True)
dt = (time.time() - t) / n_rows
print(f'{dt * 1e6:.0f} мкс/строка -> {dt * 39.4e6 / 60:.0f} мин на тесте (Colab CPU, не их железо)')
print('наш np_score с ONNX:', np_score(Pn, Yva, Mva))
print('наш evaluate с PyTorch:', {k: round(v[0], 3) for k, v in evaluate(model if False else net.to(dev), Xva, Yva, Mva).items()})

/tmp/ipykernel_322/4013712005.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Step(net), (torch.zeros(1, 1, 112), torch.zeros(1, 1, 128), torch.zeros(1, 1, 128)),


total 1.5M
-rw-r--r-- 1 root root 755K Oct 31 19:49 model.onnx
-rw-r--r-- 1 root root 3.0K Oct 31 19:49 solution.py
-rw-r--r-- 1 root root 700K Oct 31 19:49 submission.zip
0 2 с
25 55 с
50 108 с
75 159 с
105 мкс/строка -> 69 мин на тесте (Colab CPU, не их железо)
наш np_score с ONNX: {'все строки': (0.446, array([0.456, 0.436])), 'is_scored': (0.599, array([0.598, 0.599]))}
наш evaluate с PyTorch: {'все строки': 0.446, 'is_scored': 0.599}


In [ ]:
import importlib.util, time, numpy as np
from types import SimpleNamespace

def load_sol(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
SOL = {'baseline': load_sol('/content/wnn_connectome_starterpack/baseline/solution.py', 'sol_base'),
       'наша': load_sol('/content/sub/solution.py', 'sol_ours')}

def speed(sol, n_seq=12):
    mdl = sol.PredictionModel(); t = time.time()
    for s in range(n_seq):
        Xs = np.ascontiguousarray(Xva[s])
        for st in range(L):
            mdl.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= WARM, state=Xs[st]))
    return (time.time() - t) / (n_seq * L) * 1e6

for rnd in range(2):
    for name, sol in SOL.items():
        print(f'раунд {rnd + 1}: {name:9s} {speed(sol):.0f} мкс/строка', flush=True)

раунд 1: baseline  92 мкс/строка
раунд 1: наша      99 мкс/строка
раунд 2: baseline  95 мкс/строка
раунд 2: наша      99 мкс/строка


In [ ]:
import os, sys, subprocess, numpy as np, torch, torch.nn as nn

src = open('/content/wnn_connectome_starterpack/baseline/solution.py').read()
for hid in (128, 192, 256):
    d = f'/content/bench/h{hid}'; os.makedirs(d, exist_ok=True)
    net, z = Net(hid).eval(), torch.zeros(1, 1, hid)
    torch.onnx.export(StepH(net), (torch.zeros(1, 1, 112), z, z.clone()), f'{d}/model.onnx',
                      input_names=['features', 'hidden_0', 'hidden_1'],
                      output_names=['prediction', 'hidden_0_out', 'hidden_1_out'], opset_version=17, dynamo=False)
    open(f'{d}/solution.py', 'w').write(src.replace('baseline.onnx', 'model.onnx').replace('128', str(hid)))

open('/content/bench_run.py', 'w').write('''
import os, sys, time
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
from types import SimpleNamespace
sys.path.insert(0, sys.argv[1])
import solution
X = np.load("/content/data/valid_X.npy", mmap_mode="r")
m = solution.PredictionModel(); n = 6; t = time.time()
for s in range(n):
    Xs = np.ascontiguousarray(X[s])
    for st in range(20000):
        m.predict(SimpleNamespace(seq_ix=s, step_in_seq=st, need_prediction=st >= 99, state=Xs[st]))
print((time.time() - t) / (n * 20000) * 1e6)
''')

times = {128: [], 192: [], 256: []}
for rnd in range(3):
    for hid in times:
        r = subprocess.run([sys.executable, '/content/bench_run.py', f'/content/bench/h{hid}'], capture_output=True, text=True)
        if r.returncode: print(r.stderr[-500:]); raise SystemExit
        times[hid].append(float(r.stdout)); print(f'раунд {rnd + 1}: h={hid} {times[hid][-1]:.0f} мкс/строка', flush=True)

med = {h: float(np.median(v)) for h, v in times.items()}
for h in med:
    print(f'h={h}: {med[h]:.0f} мкс, {med[h] / med[128]:.2f}× от h=128 -> оценка на их машине ~{28.5 * med[h] / med[128]:.0f} мин')

/tmp/ipykernel_322/3180597289.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(StepH(net), (torch.zeros(1, 1, 112), z, z.clone()), f'{d}/model.onnx',


раунд 1: h=128 105 мкс/строка
раунд 1: h=192 145 мкс/строка
раунд 1: h=256 196 мкс/строка
раунд 2: h=128 95 мкс/строка
раунд 2: h=192 155 мкс/строка
раунд 2: h=256 188 мкс/строка
раунд 3: h=128 102 мкс/строка
раунд 3: h=192 141 мкс/строка
раунд 3: h=256 202 мкс/строка
h=128: 102 мкс, 1.00× от h=128 -> оценка на их машине ~28 мин
h=192: 145 мкс, 1.43× от h=128 -> оценка на их машине ~41 мин
h=256: 196 мкс, 1.93× от h=128 -> оценка на их машине ~55 мин


In [ ]:
# 1. Больше валидационных последовательностей (~1 минута)
build('valid', 'valid300', 300, 1, True)
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
print('доля is_scored:', Mva[:, WARM:].mean().round(3))

# 2. Текущая модель h=128 на новой валидации
net128 = Net(128).to(dev); net128.load_state_dict(torch.load(CKPT, map_location=dev))
r128 = evaluate(net128, Xva, Yva, Mva)
print('h=128 на valid300:', {k: round(v[0], 3) for k, v in r128.items()})

# 3. h=192 на тех же 3000 последовательностях, те же 8 эпох
Xtr = np.load('/content/data/train3k_X.npy', mmap_mode='r'); Ytr = np.load('/content/data/train3k_Y.npy')
torch.manual_seed(0); m192 = Net(192).to(dev)
print('параметров:', sum(p.numel() for p in m192.parameters()))
m192 = train(m192, epochs=8, tag='gru_w3k_h192')

valid300 0 0 с
valid300 100 16 с
valid300 200 28 с
доля is_scored: 0.095
h=128 на valid300: {'все строки': 0.488, 'is_scored': 0.665}
параметров: 398978
эпоха 1/8 | 383 с | loss 1.0522 | valid все строки 0.431 [0.434 0.427] | is_scored 0.641 [0.64  0.641]
эпоха 2/8 | 390 с | loss 1.0073 | valid все строки 0.473 [0.475 0.47 ] | is_scored 0.648 [0.648 0.649]
эпоха 3/8 | 388 с | loss 0.9978 | valid все строки 0.461 [0.469 0.453] | is_scored 0.644 [0.647 0.642]
эпоха 4/8 | 388 с | loss 0.9868 | valid все строки 0.479 [0.478 0.48 ] | is_scored 0.659 [0.656 0.662]
эпоха 5/8 | 388 с | loss 0.9735 | valid все строки 0.485 [0.485 0.486] | is_scored 0.658 [0.654 0.662]
эпоха 6/8 | 388 с | loss 0.9509 | valid все строки 0.484 [0.49  0.479] | is_scored 0.661 [0.658 0.665]
эпоха 7/8 | 387 с | loss 0.9269 | valid все строки 0.483 [0.486 0.481] | is_scored 0.660 [0.657 0.664]
эпоха 8/8 | 387 с | loss 0.9099 | valid все строки 0.483 [0.485 0.48 ] | is_scored 0.659 [0.655 0.662]


In [ ]:
ck = '/content/drive/MyDrive/wnn/gru_w3k_h192_best.pt'
if not os.path.exists(ck): ck = '/content/gru_w3k_h192_best.pt'
m192.load_state_dict(torch.load(ck, map_location=dev))
r192 = evaluate(m192, Xva, Yva, Mva)
for name, r in (('h=128', r128), ('h=192', r192)):
    print(name, {k: (round(v[0], 3), v[1]) for k, v in r.items()})

h=128 {'все строки': (0.488, array([0.489, 0.487])), 'is_scored': (0.665, array([0.661, 0.67 ]))}
h=192 {'все строки': (0.484, array([0.49 , 0.479])), 'is_scored': (0.661, array([0.658, 0.665]))}


In [ ]:
UN = np.array([0.1254, 0.2413, 0.4685, 0.6569])

def fam_of(Y):
    a = np.abs(np.clip(Y[..., 0], -2, 2)).astype(np.float64)
    mn = np.where(a > 0, a, 9.0).min(1)                    # наименьшее ненулевое |t0| в последовательности = единица
    f = np.abs(mn[:, None] - UN[None]).argmin(1)
    print('макс. отклонение от ближайшей единицы:', np.abs(mn - UN[f]).max().round(4), '| классы:', np.bincount(f, minlength=4))
    return f

def augment(src, dst, fam):
    X = np.load(f'/content/data/{src}_X.npy', mmap_mode='r')
    out = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (len(X), L, NF + 4))
    for s in range(0, len(X), 20):
        out[s:s + 20, :, :NF] = X[s:s + 20]
        out[s:s + 20, :, NF:] = np.eye(4, dtype=np.float32)[fam[s:s + 20]][:, None, :]
    out.flush()

Y6 = np.load('/content/data/train_Y.npy'); Y3 = np.load('/content/data/valid300_Y.npy')
augment('train', 'train_f', fam_of(Y6)); augment('valid300', 'valid300_f', fam_of(Y3))

class Net2(nn.Module):
    def __init__(self, nin, h=128):
        super().__init__(); self.gru = nn.GRU(nin, h, num_layers=2, batch_first=True); self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h); return self.head(y), h

def ld(n): return np.load(f'/content/data/{n}_X.npy', mmap_mode='r')
Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
def run(tag, Xt, Xv, nin, seed):
    global Xtr, Ytr, Xva
    Xtr, Ytr, Xva = Xt, Y6, Xv
    torch.manual_seed(seed); m = train(Net2(nin).to(dev), epochs=8, tag=tag)
    r = evaluate(m, Xva, Yva, Mva)                          # последняя эпоха, без выбора по валидации
    return {k: round(v[0], 3) for k, v in r.items()}

res = {}
for seed in (0, 1):
    res[('контроль', seed)] = run('ctl', ld('train'), ld('valid300'), NF, seed)
    res[('оракул', seed)] = run('orc', ld('train_f'), ld('valid300_f'), NF + 4, seed)
for k, v in res.items(): print(k, v)

макс. отклонение от ближайшей единицы: 0.0 | классы: [ 83  69 173 275]
макс. отклонение от ближайшей единицы: 0.0 | классы: [68 54 80 98]
эпоха 1/8 | 37 с | loss 1.1452 | valid все строки 0.435 [0.438 0.431] | is_scored 0.630 [0.626 0.635]
эпоха 2/8 | 34 с | loss 1.0646 | valid все строки 0.453 [0.457 0.448] | is_scored 0.630 [0.63  0.629]
эпоха 3/8 | 30 с | loss 1.0361 | valid все строки 0.461 [0.466 0.455] | is_scored 0.641 [0.641 0.642]
эпоха 4/8 | 21 с | loss 1.0211 | valid все строки 0.458 [0.454 0.462] | is_scored 0.638 [0.638 0.638]
эпоха 5/8 | 14 с | loss 0.9967 | valid все строки 0.469 [0.471 0.467] | is_scored 0.646 [0.648 0.644]
эпоха 6/8 | 20 с | loss 0.9767 | valid все строки 0.469 [0.473 0.465] | is_scored 0.652 [0.645 0.658]
эпоха 7/8 | 16 с | loss 0.9483 | valid все строки 0.469 [0.469 0.469] | is_scored 0.652 [0.647 0.657]
эпоха 8/8 | 17 с | loss 0.9412 | valid все строки 0.467 [0.467 0.467] | is_scored 0.653 [0.647 0.658]
эпоха 1/8 | 40 с | loss 1.1359 | valid все стр

In [ ]:
Y3k = np.load('/content/data/train3k_Y.npy')
f3k = fam_of(Y3k)
print('доли train3k:', np.round(np.bincount(f3k, minlength=4) / len(f3k), 3))
print('доли valid (полностью, из прошлых запусков): [0.213 0.185 0.230 0.372]')

макс. отклонение от ближайшей единицы: 0.0 | классы: [ 387  375  955 1283]
доли train3k: [0.129 0.125 0.318 0.428]
доли valid (полностью, из прошлых запусков): [0.213 0.185 0.230 0.372]


In [ ]:
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')          # без колонок оракула
fam_v = fam_of(Yva)

@torch.no_grad()
def stats(model, X, Y, M, bs=50, ch=2000):
    model.eval()
    S = {k: torch.zeros(2, 6, dtype=torch.float64, device=dev) for k in ('все строки', 'is_scored')}
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)
        y = torch.from_numpy(np.ascontiguousarray(Y[s:s + bs])).to(dev).clamp(-2, 2)
        m_sc = torch.from_numpy(np.ascontiguousarray(M[s:s + bs])).to(dev)
        h, outs = None, []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); outs.append(o)
        p = torch.cat(outs, 1).clamp(-2, 2)
        post = torch.zeros(L, dtype=torch.bool, device=dev); post[WARM:] = True
        for name, mask in (('все строки', post[None].expand_as(m_sc)), ('is_scored', m_sc & post[None])):
            for j in (0, 1):
                yy, pp = y[..., j][mask].double(), p[..., j][mask].double(); a = yy.abs()
                S[name][j] += torch.stack([a.sum(), (a*yy).sum(), (a*pp).sum(), (a*yy*yy).sum(), (a*pp*pp).sum(), (a*yy*pp).sum()])
    return {k: v.cpu().numpy() for k, v in S.items()}

def corr(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    cov = T[:, 5] / sw - my * mp
    den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
    return float(np.mean(np.where(den > 1e-16, cov / np.sqrt(np.maximum(den, 1e-16)), 0.0)))

SF = {}
for f in range(4):
    ix = np.where(fam_v == f)[0]
    SF[f] = stats(net128, Xva[ix], Yva[ix], Mva[ix])
    print(f'семейство {f} ({UN[f]}), {len(ix)} посл.: все строки {corr(SF[f]["все строки"]):.3f} | is_scored {corr(SF[f]["is_scored"]):.3f}')

have = np.bincount(fam_v, minlength=4) / len(fam_v)
mixes = {'валидационная смесь': np.array([0.2125, 0.1853, 0.2301, 0.3721]),
         'смесь train3k': np.array([0.129, 0.125, 0.318, 0.428]),
         'как в valid300': have}
for name, mix in mixes.items():
    m = mix / have
    out = {k: round(corr(sum(m[f] * SF[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
    print(f'{name:22s}', out)

макс. отклонение от ближайшей единицы: 0.0 | классы: [68 54 80 98]
семейство 0 (0.1254), 68 посл.: все строки 0.561 | is_scored 0.772
семейство 1 (0.2413), 54 посл.: все строки 0.479 | is_scored 0.743
семейство 2 (0.4685), 80 посл.: все строки 0.335 | is_scored 0.702
семейство 3 (0.6569), 98 посл.: все строки 0.235 | is_scored 0.396
валидационная смесь    {'все строки': 0.487, 'is_scored': 0.653}
смесь train3k          {'все строки': 0.444, 'is_scored': 0.61}
как в valid300         {'все строки': 0.488, 'is_scored': 0.665}


In [ ]:
SAT, U3 = 99, 0.6569
PAIRS = [(0,-1),(-1,0),(-1,-1),(-3,1),(1,-3),(-3,2),(2,-3),(-3,3),(3,-3),(-3,SAT),(SAT,-3)]
def code3(y):
    k = np.round(y / U3).astype(np.int32); k[y >= 2 - 1e-6] = SAT; k[y <= -2 + 1e-6] = -SAT; return k

@torch.no_grad()
def predict(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s + bs]])).to(dev); h = None; o_ = []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

def wc(y, p, m):
    y, p = y[m].astype(np.float64), p[m].astype(np.float64); w = np.abs(y); sw = w.sum()
    my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    cov, vy, vp = (w * (y - my) * (p - mp)).sum(), (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
    return cov / np.sqrt(vy * vp) if vp > 1e-12 else 0.0

ix4 = np.where(fam_v == 3)[0]
P4 = predict(net128, Xva, ix4); Y4 = np.clip(Yva[ix4], -2, 2).astype(np.float64); M4 = Mva[ix4]
post = np.zeros(L, bool); post[WARM:] = True
allm, sel = np.broadcast_to(post, M4.shape), M4 & post
print('семейство 4, проверка (ждём 0.235 / 0.396):',
      [round(np.mean([wc(Y4[..., j], P4[..., j], m) for j in (0, 1)]), 3) for m in (allm, sel)])

k0, k1 = code3(Y4[..., 0]), code3(Y4[..., 1])
cls = np.full(k0.shape, len(PAIRS), np.int16)
for c, (a, b) in enumerate(PAIRS): cls[(k0 == a) & (k1 == b)] = c
contrib = []
for j in (0, 1):
    y = Y4[..., j][sel]; w = np.abs(y); m = (w * y).sum() / w.sum()
    cj = np.zeros(cls.shape); cj[sel] = w * (y - m) ** 2; contrib.append(cj / cj.sum())
print('is_scored; класс | доля строк | доля взвеш. дисперсии t0, t1 | среднее y0, ŷ0 | y1, ŷ1')
for c in range(len(PAIRS) + 1):
    g = (cls == c) & sel
    if g.sum() == 0: continue
    nm = PAIRS[c] if c < len(PAIRS) else 'другое'
    print(f'{str(nm):10s} {g.sum() / sel.sum():6.3f} | {contrib[0][g].sum():.3f} {contrib[1][g].sum():.3f} | '
          f'{Y4[..., 0][g].mean():6.2f} {P4[..., 0][g].mean():6.2f} | {Y4[..., 1][g].mean():6.2f} {P4[..., 1][g].mean():6.2f}')

семейство 4, проверка (ждём 0.235 / 0.396): [np.float64(0.235), np.float64(0.396)]
is_scored; класс | доля строк | доля взвеш. дисперсии t0, t1 | среднее y0, ŷ0 | y1, ŷ1
(0, -1)     0.352 | 0.000 0.003 |   0.00  -0.34 |  -0.66  -0.97
(-1, 0)     0.325 | 0.001 0.000 |  -0.66  -0.88 |   0.00  -0.39
(-1, -1)    0.009 | 0.000 0.000 |  -0.66  -0.66 |  -0.66  -0.81
(-3, 1)     0.085 | 0.209 0.095 |  -1.97  -0.99 |   0.66  -0.31
(1, -3)     0.091 | 0.094 0.202 |   0.66  -0.29 |  -1.97  -1.05
(-3, 2)     0.032 | 0.080 0.153 |  -1.97  -0.91 |   1.31  -0.41
(2, -3)     0.038 | 0.172 0.085 |   1.31  -0.37 |  -1.97  -0.99
(-3, 3)     0.021 | 0.052 0.258 |  -1.97  -0.82 |   1.97  -0.56
(3, -3)     0.015 | 0.181 0.034 |   1.97  -0.32 |  -1.97  -1.04
(-3, 99)    0.010 | 0.024 0.122 |  -1.97  -0.87 |   2.00  -0.46
(99, -3)    0.014 | 0.175 0.032 |   2.00  -0.34 |  -1.97  -1.02
другое      0.008 | 0.013 0.013 |  -0.53  -0.65 |  -0.61  -0.75


In [ ]:
VALMIX = np.array([0.2125, 0.1853, 0.2301, 0.3721])
def mix_score(model):
    Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]; Sm[f] = stats(model, Xva[ix], Yva[ix], Mva[ix])
    m = VALMIX / have
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
print('h=128, валидационная смесь:', mix_score(net128))   # ждём 0.487 / 0.653

h=128, валидационная смесь: {'все строки': 0.487, 'is_scored': 0.653}


In [ ]:
side = np.sign(k1 - k0)                                   # какой из двух таргетов отрицательный: -1 это t0, +1 это t1, 0 это оба (-1,-1)
calm = np.isin(cls, [0, 1, 2])
groups = {'только сторона': side + 1,
          'сторона + «событие или нет»': (side + 1) * 2 + (~calm),
          'полный класс пары': cls}

def ladder_pred(gid, kind):
    P = np.zeros(Y4.shape)
    for j in (0, 1):
        y = Y4[..., j]; src = np.abs(y) * y; w = np.abs(y)
        for g in np.unique(gid[sel]):
            m = sel & (gid == g)
            P[..., j][m] = (src[m].sum() / w[m].sum()) if kind == 'oracle' else P4[..., j][m].mean()
    return P

score = lambda P: round(np.mean([wc(Y4[..., j], P[..., j], sel) for j in (0, 1)]), 3)
print('наша модель:', score(P4))
for name, gid in groups.items():
    print(f'{name:30s} средний ŷ нашей модели по группе: {score(ladder_pred(gid, "model")):.3f} | потолок: {score(ladder_pred(gid, "oracle")):.3f}')

наша модель: 0.396
только сторона                 средний ŷ нашей модели по группе: 0.881 | потолок: 0.881


/tmp/ipykernel_322/3567509263.py:13: RuntimeWarning: invalid value encountered in scalar divide
  P[..., j][m] = (src[m].sum() / w[m].sum()) if kind == 'oracle' else P4[..., j][m].mean()


сторона + «событие или нет»    средний ŷ нашей модели по группе: 0.922 | потолок: 0.000
полный класс пары              средний ŷ нашей модели по группе: 0.875 | потолок: 0.000


In [ ]:
from sklearn.metrics import roc_auc_score

low0 = k0 < k1                                    # t0 оказался «нижним»; k0 == k1 это класс (-1,-1)
d = P4[..., 0] - P4[..., 1]                       # d < 0 -> модель считает, что нижний это t0
valid_side = sel & (k0 != k1)
ev = np.isin(cls, np.arange(3, 11)); calm = np.isin(cls, [0, 1])

def acc(m): return np.mean((d[m] < 0) == low0[m])
print(f'точность стороны, все строки is_scored: {acc(valid_side):.3f} (строк {valid_side.sum():,})')
print(f'  на событиях {acc(valid_side & ev):.3f} | на спокойных {acc(valid_side & calm):.3f}')
print(f'AUC стороны: {roc_auc_score(low0[valid_side], -d[valid_side]):.3f}')

ad = np.abs(d[valid_side]); q = np.quantile(ad, [1/3, 2/3])
for name, m in [('малая |d|', ad <= q[0]), ('средняя', (ad > q[0]) & (ad <= q[1])), ('большая |d|', ad > q[1])]:
    print(f'  точность при {name}: {np.mean(((d[valid_side] < 0) == low0[valid_side])[m]):.3f}')

def two_group(gid):                               # двузначный предсказатель: значения не важны, важно разбиение
    P = np.zeros(Y4.shape)
    for j in (0, 1):
        y = Y4[..., j]; w = np.abs(y)
        for g in np.unique(gid[sel]):
            m = sel & (gid == g); sw = w[m].sum()
            P[..., j][m] = (w[m] * y[m]).sum() / sw if sw > 0 else 0.0
    return P
sc = lambda P: round(np.mean([wc(Y4[..., j], P[..., j], sel) for j in (0, 1)]), 3)
print('скор: нашей модели', sc(P4), '| знаку d (жёсткая сторона)', sc(two_group((d < 0).astype(int))),
      '| истинной стороне (потолок)', sc(two_group(low0.astype(int))))

точность стороны, все строки is_scored: 0.775 (строк 238,220)
  на событиях 0.737 | на спокойных 0.790
AUC стороны: 0.792
  точность при малая |d|: 0.753
  точность при средняя: 0.761
  точность при большая |d|: 0.810
скор: нашей модели 0.396 | знаку d (жёсткая сторона) 0.348 | истинной стороне (потолок) 0.875


In [ ]:
X3 = np.load('/content/data/train3k_X.npy', mmap_mode='r')
ix_tr = np.where(f3k == 3)[0][:98]                         # 98 обучающих последовательностей семейства 4, как и валидационных
Ptr = predict(net128, X3, ix_tr)
Ytr4 = np.clip(Y3k[ix_tr], -2, 2).astype(np.float64)

def side_report(P, Y, name):
    post_m = np.broadcast_to(post, Y.shape[:2])
    k0, k1 = code3(Y[..., 0]), code3(Y[..., 1])
    ev = post_m & (np.minimum(k0, k1) <= -3)               # строки-события: у одного из таргетов «-3»
    low0 = k0 < k1; d = P[..., 0] - P[..., 1]
    a = np.mean((d[ev] < 0) == low0[ev]); u = roc_auc_score(low0[ev], -d[ev])
    c = np.mean([wc(Y[..., j], P[..., j], post_m) for j in (0, 1)])
    print(f'{name:22s} событий {ev.sum():>9,} | точность стороны {a:.3f} | AUC {u:.3f} | скор по всем строкам {c:.3f}')

side_report(Ptr, Ytr4, 'train (3000 посл.)')
side_report(P4, Y4, 'valid (valid300)')

train (3000 посл.)     событий   627,548 | точность стороны 0.632 | AUC 0.693 | скор по всем строкам 0.302
valid (valid300)       событий   380,905 | точность стороны 0.621 | AUC 0.668 | скор по всем строкам 0.235


In [ ]:
Xtr = np.load('/content/data/train_X.npy', mmap_mode='r'); Ytr = Y6
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r')      # Yva, Mva, fam_v, have, VALMIX, post уже в ядре

def sums_np(P, Y, M):
    post_m = np.broadcast_to(post, M.shape); out = {}
    for name, m in (('все строки', post_m), ('is_scored', M & post_m)):
        T = np.zeros((2, 6))
        for j in (0, 1):
            y = np.clip(Y[..., j][m], -2, 2).astype(np.float64); p = np.clip(P[..., j][m], -2, 2).astype(np.float64); a = np.abs(y)
            T[j] = [a.sum(), (a*y).sum(), (a*p).sum(), (a*y*y).sum(), (a*p*p).sum(), (a*y*p).sum()]
        out[name] = T
    return out

def ens_mix(models):
    m = VALMIX / have; Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        P = np.mean([predict(mod, Xva, ix) for mod in models], axis=0)
        Sm[f] = sums_np(P, Yva[ix], Mva[ix])
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}

models = {}
for ch in (1000, 4000):
    for seed in (0, 1):
        torch.manual_seed(seed); mdl = Net().to(dev)
        models[(ch, seed)] = train(mdl, epochs=8, ch=ch, bs=(8 if ch == 4000 else 32), tag=f'e_{ch}_{seed}')
        print(f'>>> ch={ch} сид {seed}:', ens_mix([models[(ch, seed)]]), flush=True)
for ch in (1000, 4000):
    print(f'>>> ансамбль из 2 сидов, ch={ch}:', ens_mix([models[(ch, 0)], models[(ch, 1)]]))

эпоха 1/8 | 34 с | loss 1.1452 | valid все строки 0.435 [0.438 0.431] | is_scored 0.630 [0.626 0.635]
эпоха 2/8 | 32 с | loss 1.0646 | valid все строки 0.453 [0.457 0.448] | is_scored 0.630 [0.63  0.629]
эпоха 3/8 | 33 с | loss 1.0361 | valid все строки 0.461 [0.466 0.455] | is_scored 0.641 [0.641 0.642]
эпоха 4/8 | 32 с | loss 1.0211 | valid все строки 0.458 [0.454 0.462] | is_scored 0.638 [0.638 0.638]
эпоха 5/8 | 33 с | loss 0.9967 | valid все строки 0.469 [0.471 0.467] | is_scored 0.646 [0.648 0.644]
эпоха 6/8 | 32 с | loss 0.9767 | valid все строки 0.469 [0.473 0.465] | is_scored 0.652 [0.645 0.658]
эпоха 7/8 | 34 с | loss 0.9483 | valid все строки 0.469 [0.469 0.469] | is_scored 0.652 [0.647 0.657]
эпоха 8/8 | 33 с | loss 0.9412 | valid все строки 0.467 [0.467 0.467] | is_scored 0.653 [0.647 0.658]
>>> ch=1000 сид 0: {'все строки': 0.467, 'is_scored': 0.64}
эпоха 1/8 | 34 с | loss 1.1420 | valid все строки 0.434 [0.437 0.43 ] | is_scored 0.628 [0.624 0.632]
эпоха 2/8 | 38 с | los

In [ ]:
!rm -f /content/data/train_f_X.npy /content/data/valid300_f_X.npy
!rm -rf /content/bench; df -h /content | tail -1

def augment_diff(src, dst):
    X = np.load(f'/content/data/{src}_X.npy', mmap_mode='r')
    out = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (len(X), L, 2 * NF))
    for s in range(0, len(X), 20):
        b = np.asarray(X[s:s + 20])
        out[s:s + 20, :, :NF] = b
        out[s:s + 20, :, NF:] = np.diff(b, axis=1, prepend=b[:, :1])    # на шаге 0 разность 0, как и будет в инференсе
    out.flush()
augment_diff('train', 'train_d'); augment_diff('valid300', 'valid300_d')

Xtr = np.load('/content/data/train_d_X.npy', mmap_mode='r'); Ytr = Y6
Xva = np.load('/content/data/valid300_d_X.npy', mmap_mode='r')
models_d = {}
for seed in (0, 1):
    torch.manual_seed(seed)
    models_d[seed] = train(Net2(2 * NF).to(dev), epochs=8, tag=f'd_{seed}')
    print(f'>>> разности, сид {seed}:', ens_mix([models_d[seed]]), flush=True)

overlay         113G   72G   41G  64% /
эпоха 1/8 | 77 с | loss 1.1339 | valid все строки 0.442 [0.447 0.438] | is_scored 0.640 [0.635 0.646]
эпоха 2/8 | 69 с | loss 1.0554 | valid все строки 0.449 [0.457 0.44 ] | is_scored 0.636 [0.636 0.636]
эпоха 3/8 | 67 с | loss 1.0213 | valid все строки 0.449 [0.454 0.445] | is_scored 0.637 [0.639 0.634]
эпоха 4/8 | 66 с | loss 1.0084 | valid все строки 0.456 [0.454 0.458] | is_scored 0.638 [0.639 0.638]
эпоха 5/8 | 65 с | loss 0.9798 | valid все строки 0.465 [0.465 0.466] | is_scored 0.650 [0.648 0.651]
эпоха 6/8 | 65 с | loss 0.9478 | valid все строки 0.460 [0.465 0.455] | is_scored 0.651 [0.645 0.657]
эпоха 7/8 | 66 с | loss 0.9040 | valid все строки 0.456 [0.456 0.456] | is_scored 0.646 [0.641 0.651]
эпоха 8/8 | 66 с | loss 0.8819 | valid все строки 0.454 [0.454 0.455] | is_scored 0.646 [0.64  0.652]
>>> разности, сид 0: {'все строки': 0.454, 'is_scored': 0.633}
эпоха 1/8 | 66 с | loss 1.1341 | valid все строки 0.436 [0.445 0.427] | is_scored

In [ ]:
class AsF16:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i]).astype(np.float16).astype(np.float32)
Xva_orig = Xva; Xva = AsF16(Xva_orig)
print('h=128, float16-вход:', mix_score(net128), '(в float32 было 0.487 / 0.653)')
Xva = Xva_orig

RuntimeError: input.size(-1) must be equal to input_size. Expected 112, got 224

In [ ]:
import importlib.util, numpy as np
from types import SimpleNamespace
spec = importlib.util.spec_from_file_location('sol_chk', '/content/sub/solution.py')
sol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)
Xs = np.ascontiguousarray(Xva[0][:3000])
def run():
    m = sol.PredictionModel(); out = []
    for st in range(len(Xs)):
        r = m.predict(SimpleNamespace(seq_ix=0, step_in_seq=st, need_prediction=st >= 99, state=Xs[st]))
        if r is not None: out.append(r)
    return np.array(out)
a, b = run(), run()
print('побитово одинаковы:', np.array_equal(a, b), '| конечны:', np.isfinite(a).all())